## Taxi Data

### Setup   

In [17]:
# libraries
import matplotlib.pyplot as plt ## for basic plotting
import matplotlib as mpl ## for setting default parameters
import pandas as pd ## always
import os ## for handling file paths 1
from pathlib import Path ## for handling file paths 2
import numpy as np ## for numerical operations
import seaborn as sns ## for more advanced plotting

In [18]:
# paths
base_dir = Path("/Users/hannahmaihojgaard/Documents/GitHub/datascience2026")
data_path = Path(base_dir / "data")

### Load in data

In [19]:
df_2026 = pd.read_parquet(data_path / "yellow_tripdata_2026-01.parquet")
df_2025 = pd.read_parquet(data_path / "yellow_tripdata_2025-01.parquet")
df_2026.head()
df_2025.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


### Preproccessing

In [20]:
df_2026.shape
df_2026.columns
df_2026.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [21]:
df_2025.shape
df_2025.columns
df_2025.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


- *VendorID*: Taxi technology provider (1 = CMT, 2 = VeriFone)
- *tpep_pickup_datetime*: Timestamp when the trip started (meter on)
- *tpep_dropoff_datetime*: Timestamp when the trip ended (meter off)
- *passenger_count*: Number of passengers (driver-reported)
- *trip_distance*: Trip distance in miles (from taximeter)
- *RatecodeID*: Rate type (e.g., standard, JFK, Newark, etc.)
- *store_and_fwd_flag*: Whether trip data was stored before sending (Y/N)
- *PULocationID*: Pickup location ID (zone-based)
- *DOLocationID*: Dropoff location ID (zone-based)
- *payment_type*: Payment method (1=card, 2=cash, etc.)
- *fare_amount*: Base fare (time + distance)
- *extra*: Additional charges (e.g., rush hour, overnight)
- *mta_tax*: Fixed $0.50 MTA tax
- *tip_amount*: Tip (only recorded for card payments)
- *tolls_amount*: Total toll charges
- *improvement_surcharge*: $0.30 improvement fee
- *total_amount*: Total charged (excluding cash tips)
- *congestion_surcharge*: Fee for trips in congestion zones
- *Airport_fee*: Extra fee for airport trips
- *cbd_congestion_fee*: Central business district congestion fee


In [22]:
df_2025.isna().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64

In [23]:
# converting pickup and dropoff datetime columns to datetime format
df_2025["tpep_pickup_datetime"] = pd.to_datetime(df_2025["tpep_pickup_datetime"])
df_2025["tpep_dropoff_datetime"] = pd.to_datetime(df_2025["tpep_dropoff_datetime"])

# calculating trip duration in minutes
df_2025["trip_duration"] = (
    df_2025["tpep_dropoff_datetime"] - df_2025["tpep_pickup_datetime"]
).dt.total_seconds() / 60  # minutes 

In [24]:
# checking for negative or zero values in trip_distance, fare_amount, and trip_duration
df_2025[df_2025["trip_distance"] <= 0].shape
df_2025[df_2025["fare_amount"] <= 0].shape
df_2025[df_2025["trip_duration"] <= 0].shape

(2051, 21)

In [25]:
df_2025[df_2025["trip_distance"] <= 0].head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration
92,1,2025-01-01 00:49:48,2025-01-01 00:49:48,1.0,0.0,1.0,Y,87,264,2,...,0.0,0.0,0.00,0.00,0.0,20.06,0.0,0.0,0.0,0.000000
204,2,2025-01-01 00:37:43,2025-01-01 00:37:53,1.0,0.0,5.0,N,148,148,1,...,0.0,0.0,2.00,0.00,1.0,17.50,2.5,0.0,0.0,0.166667
358,2,2025-01-01 00:57:08,2025-01-01 00:57:16,3.0,0.0,5.0,N,141,141,1,...,0.0,0.0,0.00,0.00,1.0,33.50,2.5,0.0,0.0,0.133333
505,1,2025-01-01 00:27:40,2025-01-01 00:59:30,1.0,0.0,1.0,N,168,76,1,...,0.0,0.5,0.00,6.94,1.0,58.94,0.0,0.0,0.0,31.833333
619,2,2025-01-01 00:56:49,2025-01-01 00:56:54,4.0,0.0,5.0,N,164,164,1,...,0.0,0.0,7.05,0.00,1.0,30.55,2.5,0.0,0.0,0.083333
